In [108]:
from pathlib import Path

import pandas as pd
import numpy as np
import json
import zipfile
import tarfile

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(PUBLIC_RAW)

/Users/uriel/Documents/machine_learning/data/public_v2/raw


In [109]:
summary = pd.read_csv(
    PUBLIC_RAW
    / "_manifests"
    / "dataset_copy_summary.csv"
)

manifest = pd.read_csv(
    PUBLIC_RAW
    / "_manifests"
    / "dataset_copy_manifest.csv"
)

display(summary)

,dataset_id,priority,similarity,source_folder,dest_folder,status,selected_files,selected_bytes,selected_size,note
0,AMUSED,P1,ALTA,NaN,/Users/uriel/Documents/machine_learning/data/public_v2/raw/01_amused,PENDING_ACCESS,0,0,0 B,"Pendiente de acceso; si después aparece en la carpeta fuente, este mismo script lo copiará."
1,ZENODO_21891322,P1,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/Zenodo 21891322,/Users/uriel/Documents/machine_learning/data/public_v2/raw/02_zenodo_21891322,COPIED_OR_PRESENT,14,7611541,7.26 MB,NaN
2,VT_29042675,P1,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/29042675,/Users/uriel/Documents/machine_learning/data/public_v2/raw/03_29042675,COPIED_OR_PRESENT,47,2940689,2.80 MB,NaN
3,CATCHING_UX_FLAWS,P1,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/Catching UX Flaws,/Users/uriel/Documents/machine_learning/data/public_v2/raw/04_catching_ux_flaws,COPIED_OR_PRESENT,92,1396075,1.33 MB,"Conserva README, notebook y results/*.json; omite graphs/ y static/."
4,SERENE,P1,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/SERENE,/Users/uriel/Documents/machine_learning/data/public_v2/raw/05_serene,COPIED_OR_PRESENT,4,5423521,5.17 MB,NaN
5,P3M,P2,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/P3M,/Users/uriel/Documents/machine_learning/data/public_v2/raw/06_p3m,COPIED_OR_PRESENT,23,27151631,25.89 MB,NaN
6,PHR,P2,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/PHR,/Users/uriel/Documents/machine_learning/data/public_v2/raw/07_phr,COPIED_OR_PRESENT,2,66454,64.90 KB,NaN
7,DASHBOARDS,P2,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/Dashboards,/Users/uriel/Documents/machine_learning/data/public_v2/raw/08_dashboards,COPIED_OR_PRESENT,3,27219727,25.96 MB,Interaction Data.zip se mantiene comprimido en raw.
8,CULTURAL_HERITAGE,P2,ALTA,NaN,/Users/uriel/Documents/machine_learning/data/public_v2/raw/09_cultural_heritage,MISSING_DATASET_FOLDER,0,0,0 B,"Si no existe la carpeta del dataset y solo está el PDF de Hoang, se marca como referencia."
9,UICRIT_RICO,P2,ALTA,/Users/uriel/Downloads/CONJUNTO DE DATOS PARA LA TESIS/Version 2/RICO + uicrit,/Users/uriel/Documents/machine_learning/data/public_v2/raw/10_uicrit_rico,COPIED_OR_PRESENT,7,6500008688,6.05 GB,Por defecto no copia traces.tar.gz; usa --include-rico-traces para incluirlo.


In [110]:
summary[
    [
        "dataset_id",
        "priority",
        "similarity",
        "status",
        "selected_files",
        "selected_size"
    ]
]

,dataset_id,priority,similarity,status,selected_files,selected_size
0,AMUSED,P1,ALTA,PENDING_ACCESS,0,0 B
1,ZENODO_21891322,P1,ALTA,COPIED_OR_PRESENT,14,7.26 MB
2,VT_29042675,P1,ALTA,COPIED_OR_PRESENT,47,2.80 MB
3,CATCHING_UX_FLAWS,P1,ALTA,COPIED_OR_PRESENT,92,1.33 MB
4,SERENE,P1,ALTA,COPIED_OR_PRESENT,4,5.17 MB
5,P3M,P2,ALTA,COPIED_OR_PRESENT,23,25.89 MB
6,PHR,P2,ALTA,COPIED_OR_PRESENT,2,64.90 KB
7,DASHBOARDS,P2,ALTA,COPIED_OR_PRESENT,3,25.96 MB
8,CULTURAL_HERITAGE,P2,ALTA,MISSING_DATASET_FOLDER,0,0 B
9,UICRIT_RICO,P2,ALTA,COPIED_OR_PRESENT,7,6.05 GB


In [111]:
records = []

for path in PUBLIC_RAW.rglob("*"):

    if not path.is_file():
        continue

    rel = path.relative_to(PUBLIC_RAW)

    if rel.parts[0] in {
        "_manifests",
        "_references"
    }:
        continue

    records.append({
        "dataset_folder": rel.parts[0],
        "relative_path": str(rel),
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "size_mb": path.stat().st_size / (1024 ** 2)
    })

inventory = pd.DataFrame(records)

print(inventory.shape)

display(
    inventory.head(30)
)

(192, 6)


,dataset_folder,relative_path,filename,extension,size_bytes,size_mb
0,07_phr,07_phr/Results-PHRs-usab-sust.xlsx,Results-PHRs-usab-sust.xlsx,.xlsx,64017,0.061051
1,07_phr,07_phr/README.txt,README.txt,.txt,2437,0.002324
2,05_serene,05_serene/ground-truth.csv,ground-truth.csv,.csv,5331,0.005084
3,05_serene,05_serene/questionnaire.pdf,questionnaire.pdf,.pdf,5205041,4.963914
4,05_serene,05_serene/metrics.csv,metrics.csv,.csv,4719,0.004500
5,05_serene,05_serene/answers.json,answers.json,.json,208430,0.198774
6,06_p3m,06_p3m/1. GPT-4 - All Possible Defects per Screenshot.xlsx,1. GPT-4 - All Possible Defects per Screenshot.xlsx,.xlsx,21289,0.020303
7,06_p3m,06_p3m/2. GPT-4 - Defects Consolidation.xlsx,2. GPT-4 - Defects Consolidation.xlsx,.xlsx,45353,0.043252
8,06_p3m,06_p3m/Complete Prompt Used.pdf,Complete Prompt Used.pdf,.pdf,69434,0.066217
9,02_zenodo_21891322,02_zenodo_21891322/01.png,01.png,.png,932158,0.888975


In [112]:
inventory.groupby(
    "dataset_folder"
).agg(
    files=("filename", "count"),
    size_mb=("size_mb", "sum")
).sort_values(
    "size_mb",
    ascending=False
)

,files,size_mb
dataset_folder,,
10_uicrit_rico,7,6198.891342
08_dashboards,3,25.958755
06_p3m,23,25.893813
02_zenodo_21891322,14,7.258931
05_serene,4,5.172273
03_29042675,47,2.804460
04_catching_ux_flaws,92,1.331401
07_phr,2,0.063375


In [113]:
pd.crosstab(
    inventory["dataset_folder"],
    inventory["extension"]
)

extension,.csv,.docx,.gz,.ipynb,.json,.md,.npy,.pdf,.png,.rtf,.txt,.xlsx,.zip
dataset_folder,,,,,,,,,,,,,
02_zenodo_21891322,0,0,0,1,0,0,0,1,12,0,0,0,0
03_29042675,2,43,0,0,0,0,0,0,0,2,0,0,0
04_catching_ux_flaws,0,0,0,1,90,1,0,0,0,0,0,0,0
05_serene,2,0,0,0,1,0,0,1,0,0,0,0,0
06_p3m,0,0,0,0,0,0,0,1,20,0,0,2,0
07_phr,0,0,0,0,0,0,0,0,0,0,1,1,0
08_dashboards,1,0,0,0,0,0,0,0,0,0,0,1,1
10_uicrit_rico,3,0,1,0,1,0,1,0,0,0,1,0,0


In [114]:
def inspeccionar_csv(path, nrows=5):

    try:
        df = pd.read_csv(
            path,
            nrows=nrows,
            low_memory=False
        )

        return {
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "n_columns": len(df.columns),
            "columns": list(df.columns),
            "status": "OK"
        }

    except Exception as e:

        return {
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "n_columns": np.nan,
            "columns": None,
            "status": f"ERROR: {e}"
        }

In [115]:
csv_results = []

for path in PUBLIC_RAW.rglob("*.csv"):
    csv_results.append(
        inspeccionar_csv(path)
    )

csv_schema = pd.DataFrame(
    csv_results
)

display(csv_schema)

,file,n_columns,columns,status
0,_manifests/dataset_copy_manifest.csv,8.0,"[dataset_id, priority, similarity, source_file, relative_file, dest_file, size_bytes, action]",OK
1,_manifests/dataset_copy_summary.csv,10.0,"[dataset_id, priority, similarity, source_folder, dest_folder, status, selected_files, selected_bytes, selected_size, note]",OK
2,05_serene/ground-truth.csv,NaN,None,"ERROR: Error tokenizing data. C error: Expected 1 fields in line 4, saw 2\n"
3,05_serene/metrics.csv,10.0,"[Unnamed: 0, condition, M1, M2, M3, M4, M5, M6, M7, M8]",OK
4,10_uicrit_rico/app_details.csv,8.0,"[App Package Name, Play Store Name, Category, Average Rating, Number of Ratings, Number of Downloads, Date Updated, Icon URL]",OK
5,10_uicrit_rico/ui_details.csv,4.0,"[UI Number, App Package Name, Interaction Trace Number, UI Number in Trace]",OK
6,10_uicrit_rico/uicrit_public.csv,9.0,"[rico_id, task, aesthetics_rating, learnability, efficency, usability_rating, design_quality_rating, comments_source, comments]",OK
7,03_29042675/Prototype_Metadata.csv,4.0,"[Prototype, Prototype Type, Domain, Scource URL]",OK
8,03_29042675/IRR_Results.csv,36.0,"[Name, Prototype Link, H1 Compare, H2 Compare, H3 Compare, H4 Compare, H5 Compare, H6 Compare, H7 Compare, H8 Compare, H9 Compare, H10 Compare, Unnamed: 12, Unnamed: 13, H1. Overal Rating (4), H1,...",OK
9,08_dashboards/dependent and independent variables.csv,16.0,"[P, DB, Issue, Type, Q, Answer, Time, SGLT, mental, physical, temporal, performance, effort, frustration, avg_others, avg_all]",OK


In [116]:
def inspeccionar_json(path):

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:
            data = json.load(f)

        if isinstance(data, dict):

            structure = "dict"
            keys = list(data.keys())

        elif isinstance(data, list):

            structure = "list"

            if (
                len(data) > 0
                and
                isinstance(data[0], dict)
            ):
                keys = list(data[0].keys())
            else:
                keys = []

        else:
            structure = type(data).__name__
            keys = []

        return {
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "structure": structure,
            "top_level_keys": keys,
            "status": "OK"
        }

    except Exception as e:

        return {
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "structure": None,
            "top_level_keys": None,
            "status": f"ERROR: {e}"
        }

In [117]:
json_paths = list(
    PUBLIC_RAW.rglob("*.json")
)

print(
    "JSON encontrados:",
    len(json_paths)
)

json_schema = pd.DataFrame(
    [
        inspeccionar_json(path)
        for path in json_paths
    ]
)

display(
    json_schema.head(30)
)

JSON encontrados: 92


,file,structure,top_level_keys,status
0,05_serene/answers.json,list,"[condition, location, age, userKey, gender, internet, taranto, basilicata]",OK
1,10_uicrit_rico/ui_layout_vectors/ui_names.json,dict,[ui_names],OK
2,04_catching_ux_flaws/results/startbootstrap-creative/eval1.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
3,04_catching_ux_flaws/results/startbootstrap-creative/eval2.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
4,04_catching_ux_flaws/results/startbootstrap-creative/eval3.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
5,04_catching_ux_flaws/results/simple-product_responsive-template/eval1.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
6,04_catching_ux_flaws/results/simple-product_responsive-template/eval2.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
7,04_catching_ux_flaws/results/simple-product_responsive-template/eval3.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
8,04_catching_ux_flaws/results/material-dashboard/eval1.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK
9,04_catching_ux_flaws/results/material-dashboard/eval2.json,list,"[Heuristic, SeverityRating, IssueFound, IssueDescription, CodeReference, CodeSnippet, EvaluationAnswers, Recommendation]",OK


In [118]:
catching_json = json_schema[
    json_schema["file"].str.contains(
        "04_catching_ux_flaws"
    )
]

catching_json[
    "top_level_keys"
].astype(str).value_counts()

top_level_keys
['Heuristic', 'SeverityRating', 'IssueFound', 'IssueDescription', 'CodeReference', 'CodeSnippet', 'EvaluationAnswers', 'Recommendation']    90
Name: count, dtype: int64

In [119]:
xlsx_records = []

for path in PUBLIC_RAW.rglob("*.xlsx"):

    try:

        excel = pd.ExcelFile(path)

        xlsx_records.append({
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "sheets": excel.sheet_names,
            "n_sheets": len(
                excel.sheet_names
            ),
            "status": "OK"
        })

    except Exception as e:

        xlsx_records.append({
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "sheets": None,
            "n_sheets": np.nan,
            "status": f"ERROR: {e}"
        })

xlsx_schema = pd.DataFrame(
    xlsx_records
)

display(xlsx_schema)

,file,sheets,n_sheets,status
0,07_phr/Results-PHRs-usab-sust.xlsx,"[correlations, t-student]",2,OK
1,06_p3m/1. GPT-4 - All Possible Defects per Screenshot.xlsx,[All Possible Defects],1,OK
2,06_p3m/2. GPT-4 - Defects Consolidation.xlsx,[Defects consolidation],1,OK
3,08_dashboards/Interaction Patterns Dataset.xlsx,"[BENCHMARK, D1-Q1, D2-Q1, D3-Q2, D4-Q2, D5-Q1, D6-Q1, D7-Q2, D8-Q2, Event-element further explanati]",10,OK


In [120]:
zip_records = []

for path in PUBLIC_RAW.rglob("*.zip"):

    with zipfile.ZipFile(
        path,
        "r"
    ) as z:

        names = z.namelist()

        zip_records.append({
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "members": len(names),
            "first_files": names[:30]
        })

pd.DataFrame(
    zip_records
)

,file,members,first_files
0,08_dashboards/Interaction Data.zip,101,"[1633946848251.json, __MACOSX/, __MACOSX/._1633946848251.json, 1634049516613.json, __MACOSX/._1634049516613.json, 1635153366300.json, __MACOSX/._1635153366300.json, 1634637167060.json, __MACOSX/._..."


In [121]:
tar_records = []

for path in PUBLIC_RAW.rglob("*.tar.gz"):

    with tarfile.open(
        path,
        "r:gz"
    ) as tar:

        members = tar.getmembers()

        tar_records.append({
            "file": str(
                path.relative_to(PUBLIC_RAW)
            ),
            "members": len(members),
            "first_files": [
                m.name
                for m in members[:30]
            ]
        })

pd.DataFrame(
    tar_records
)

,file,members,first_files
0,10_uicrit_rico/unique_uis.tar.gz,132523,"[combined, combined/42231.jpg, combined/29706.jpg, combined/68068.json, combined/20331.jpg, combined/51807.jpg, combined/24825.jpg, combined/59725.json, combined/36217.json, combined/36872.jpg, co..."


In [122]:
candidate_audit = pd.DataFrame([
    {
        "dataset": "AMUSED",
        "folder": "01_amused",
        "priority": "P1",
        "similarity": "ALTA",
        "availability": "PENDING_ACCESS",
        "status": "PENDING"
    },

    {
        "dataset": "Zenodo 21891322",
        "folder": "02_zenodo_21891322",
        "priority": "P1",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "29042675",
        "folder": "03_29042675",
        "priority": "P1",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "Catching UX Flaws",
        "folder": "04_catching_ux_flaws",
        "priority": "P1",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "SERENE",
        "folder": "05_serene",
        "priority": "P1",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "P3M",
        "folder": "06_p3m",
        "priority": "P2",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "PHR",
        "folder": "07_phr",
        "priority": "P2",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "Dashboards",
        "folder": "08_dashboards",
        "priority": "P2",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    },

    {
        "dataset": "Cultural Heritage",
        "folder": "09_cultural_heritage",
        "priority": "P2",
        "similarity": "ALTA",
        "availability": "MISSING_DATASET",
        "status": "PENDING"
    },

    {
        "dataset": "UICrit + RICO",
        "folder": "10_uicrit_rico",
        "priority": "P2",
        "similarity": "ALTA",
        "availability": "AVAILABLE",
        "status": "AUDIT"
    }
])

candidate_audit

,dataset,folder,priority,similarity,availability,status
0,AMUSED,01_amused,P1,ALTA,PENDING_ACCESS,PENDING
1,Zenodo 21891322,02_zenodo_21891322,P1,ALTA,AVAILABLE,AUDIT
2,29042675,03_29042675,P1,ALTA,AVAILABLE,AUDIT
3,Catching UX Flaws,04_catching_ux_flaws,P1,ALTA,AVAILABLE,AUDIT
4,SERENE,05_serene,P1,ALTA,AVAILABLE,AUDIT
5,P3M,06_p3m,P2,ALTA,AVAILABLE,AUDIT
6,PHR,07_phr,P2,ALTA,AVAILABLE,AUDIT
7,Dashboards,08_dashboards,P2,ALTA,AVAILABLE,AUDIT
8,Cultural Heritage,09_cultural_heritage,P2,ALTA,MISSING_DATASET,PENDING
9,UICrit + RICO,10_uicrit_rico,P2,ALTA,AVAILABLE,AUDIT


In [123]:
inventory.to_csv(
    REPORTS_DIR
    / "public_v2_file_inventory.csv",
    index=False
)

candidate_audit.to_csv(
    REPORTS_DIR
    / "candidate_dataset_audit.csv",
    index=False
)

In [124]:
SERENE_DIR = PUBLIC_RAW / "05_serene"

serene_gt_path = (
    SERENE_DIR
    / "ground-truth.csv"
)

with open(
    serene_gt_path,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    for i in range(15):
        line = f.readline()

        if not line:
            break

        print(
            repr(line)
        )

'Webpage;ID;Severity;FalsePositive;InHeatmap;InAutomation;UIElement;Description\n'
'Comune di Taranto;FOOTER/LinksNotWorking;3;False;True;False;Footer;Links that are not working are still highlighted as links\n'
'Comune di Taranto;FOOTER/Typos;1;False;False;False;Footer;(UI) Various typos in the copyright section\n'
'Comune di Taranto;GALLERY/Click;4;False;False;False;Gallery;Unclear if images are clickable. And if so, are they zoomable or links?\n'
'Comune di Taranto;GALLERY/Description;4;False;False;False;Gallery;Missing photo description\n'
'Comune di Taranto;GALLERY/Margin;2;True;True;True;Gallery;(UI) Images margins and dimensions are not equal\n'
'Comune di Taranto;GALLERY/Title;2;False;True;False;Gallery Title;Text is missing margin, so it seems to overlap to buttons above. This causes confusion in the button functions\n'
'Comune di Taranto;LINKS/Buttons;3;False;True;True;Links Section;They seem like buttons but they are actually link and only the label is clickable\n'
'Comune d

In [125]:
serene_gt = pd.read_csv(
    serene_gt_path,
    sep=None,
    engine="python"
)

print(
    serene_gt.shape
)

print(
    serene_gt.columns.tolist()
)

display(
    serene_gt.head(20)
)

(46, 8)
['Webpage', 'ID', 'Severity', 'FalsePositive', 'InHeatmap', 'InAutomation', 'UIElement', 'Description']


,Webpage,ID,Severity,FalsePositive,InHeatmap,InAutomation,UIElement,Description
0,Comune di Taranto,FOOTER/LinksNotWorking,3,False,True,False,Footer,Links that are not working are still highlighted as links
1,Comune di Taranto,FOOTER/Typos,1,False,False,False,Footer,(UI) Various typos in the copyright section
2,Comune di Taranto,GALLERY/Click,4,False,False,False,Gallery,"Unclear if images are clickable. And if so, are they zoomable or links?"
3,Comune di Taranto,GALLERY/Description,4,False,False,False,Gallery,Missing photo description
4,Comune di Taranto,GALLERY/Margin,2,True,True,True,Gallery,(UI) Images margins and dimensions are not equal
5,Comune di Taranto,GALLERY/Title,2,False,True,False,Gallery Title,"Text is missing margin, so it seems to overlap to buttons above. This causes confusion in the button functions"
6,Comune di Taranto,LINKS/Buttons,3,False,True,True,Links Section,They seem like buttons but they are actually link and only the label is clickable
7,Comune di Taranto,LINKS/External,1,False,True,False,Links,External links and internal links are not differentiated
8,Comune di Taranto,LINKS/Icons,3,False,True,True,Links Section,All icons are equal
9,Comune di Taranto,NAVBAR/Design,4,False,True,True,Navbar,"Full of icons and logos, distracting. May cause overload (not minimal design)"


In [126]:
serene_gt.info()

<class 'pandas.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Webpage        46 non-null     str  
 1   ID             46 non-null     str  
 2   Severity       46 non-null     int64
 3   FalsePositive  46 non-null     bool 
 4   InHeatmap      46 non-null     bool 
 5   InAutomation   46 non-null     bool 
 6   UIElement      46 non-null     str  
 7   Description    45 non-null     str  
dtypes: bool(3), int64(1), str(4)
memory usage: 2.1 KB


In [127]:
for col in serene_gt.columns:

    print("\n", "=" * 80)
    print(col)

    print(
        serene_gt[col]
        .value_counts(
            dropna=False
        )
        .head(30)
    )


Webpage
Webpage
Regione Basilicata    25
Comune di Taranto     21
Name: count, dtype: int64

ID
ID
LINKS/External            2
NAVBAR/Location           2
NAVBAR/NoHome             2
PAGE/MissingDate          2
ACCESSIBILITY/Font        2
FOOTER/LinksNotWorking    1
FOOTER/Typos              1
GALLERY/Click             1
GALLERY/Description       1
GALLERY/Margin            1
GALLERY/Title             1
LINKS/Buttons             1
LINKS/Icons               1
NAVBAR/Design             1
NAVBAR/LinkDescription    1
NAVBAR/PopupCovers        1
NAVBAR/Size               1
NOTIFICATION/Unclear      1
OVERALL/Design            1
FOOTER/Overload           1
LINKS/Unclear             1
ALERTS/Legend             1
ALERTS/Ordering           1
ALERTS/ReadAllLink        1
ALERTS/ReadAllWhat        1
FOOTER/ImagesBtn          1
FOOTER/SmallLogos         1
GASBONUS/Click            1
GASBONUS/Links            1
GASBONUS/Overload         1
Name: count, dtype: int64

Severity
Severity
3    13
4    13

In [128]:
with open(
    SERENE_DIR / "answers.json",
    "r",
    encoding="utf-8"
) as f:

    serene_answers = json.load(f)

print(
    type(serene_answers)
)

print(
    len(serene_answers)
)

display(
    serene_answers[:2]
    if isinstance(
        serene_answers,
        list
    )
    else serene_answers
)

<class 'list'>
99


[{'condition': 'automation',
  'location': 'bari',
  'age': 21,
  'userKey': '9d91340c-5c80-49d9-8032-6f3c29ecd5b0',
  'gender': 'm',
  'internet': '7-12',
  'taranto': {'recognizedIssues': [], 'additionalIssues': []},
  'basilicata': {'recognizedIssues': [], 'additionalIssues': []}},
 {'condition': 'automation',
  'location': 'bari',
  'age': 21,
  'userKey': 'a7646798-85fa-4c69-aa3c-13f5492bbb78',
  'gender': 'm',
  'internet': '7-12',
  'taranto': {'recognizedIssues': [], 'additionalIssues': []},
  'basilicata': {'recognizedIssues': [], 'additionalIssues': []}}]

In [129]:
CATCHING_DIR = (
    PUBLIC_RAW
    / "04_catching_ux_flaws"
    / "results"
)

catching_records = []

for path in sorted(
    CATCHING_DIR.rglob("*.json")
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    site = path.parent.name

    evaluation = (
        path.stem
        .lower()
        .replace(" ", "")
    )

    for item in data:

        catching_records.append({
            "site": site,
            "evaluation": evaluation,
            "source_file": str(
                path.relative_to(
                    PUBLIC_RAW
                )
            ),

            "heuristic":
                item.get("Heuristic"),

            "severity":
                item.get("SeverityRating"),

            "issue_found":
                item.get("IssueFound"),

            "description":
                item.get(
                    "IssueDescription"
                ),

            "code_reference":
                item.get(
                    "CodeReference"
                )
        })

catching = pd.DataFrame(
    catching_records
)

print(
    catching.shape
)

display(
    catching.head(30)
)

(895, 8)


,site,evaluation,source_file,heuristic,severity,issue_found,description,code_reference
0,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Visibility of system status,2,True,No clear feedback indicators like loading spinners or success messages for user actions such as clicking buttons.,index.html (lines involving .get-started)
1,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Match between system and the real world,1,True,"The system mostly uses common terminology, but lacks iconography or UI elements that reflect real-world metaphors.",index.html (header and buttons)
2,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,User control and freedom,3,True,There is no navigation or means to undo/redo actions or return to previous states after interactions.,index.html (global navigation and footer absent)
3,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Consistency and standards,1,True,The use of inline styles for some buttons introduces minor inconsistencies with overall CSS-based layout.,"style.css (button styles), index.html (buttons)"
4,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Error prevention,2,True,"No form validation or user guidance was detected, increasing potential for user errors if forms were present.",index.html
5,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Recognition rather than recall,1,True,"The interface generally supports recognition through clear labels, but there are no tooltips or hover cues.",index.html (header and buttons)
6,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Flexibility and efficiency of use,2,True,The interface is straightforward but lacks keyboard shortcuts or personalization options for experienced users.,index.html (global layout)
7,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Aesthetic and minimalist design,1,True,"Design is minimal, though some elements (like large header) could be more balanced for improved scanability.","style.css (.blue-back, .title-task-text)"
8,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,"Help users recognize, diagnose, and recover from errors",3,True,"No error handling or messages exist for any interactions, making recovery impossible.",index.html
9,Business-Website,eval1,04_catching_ux_flaws/results/Business-Website/eval1.json,Help and documentation,4,True,"There is no accessible help section, onboarding guide, or documentation for users.",index.html (no help section or links present)


In [130]:
print(
    "Sitios:",
    catching["site"].nunique()
)

print(
    "Evaluaciones:",
    catching[
        [
            "site",
            "evaluation"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

catching[
    "heuristic"
].value_counts(
    dropna=False
)

Sitios: 30
Evaluaciones: 90


heuristic
Match between system and the real world                       86
Error prevention                                              86
Recognition rather than recall                                86
Aesthetic and minimalist design                               86
Help and documentation                                        86
Visibility of system status                                   85
User control and freedom                                      85
Consistency and standards                                     85
Flexibility and efficiency of use                             85
Help users recognize, diagnose, and recover from errors       85
1. Visibility of system status                                 2
2. Match between system and the real world                     2
3. User control and freedom                                    2
4. Consistency and standards                                   2
5. Error prevention                                            2
6. Recognition 

In [131]:
catching[
    catching[
        "heuristic"
    ]
    .astype(str)
    .str.contains(
        r"\b(?:H1|H4|H5|H6)\b",
        case=False,
        regex=True,
        na=False
    )
][
    [
        "site",
        "evaluation",
        "heuristic",
        "issue_found",
        "severity"
    ]
].head(100)

,site,evaluation,heuristic,issue_found,severity


In [132]:
catching[
    "heuristic"
].drop_duplicates().tolist()

['Visibility of system status',
 'Match between system and the real world',
 'User control and freedom',
 'Consistency and standards',
 'Error prevention',
 'Recognition rather than recall',
 'Flexibility and efficiency of use',
 'Aesthetic and minimalist design',
 'Help users recognize, diagnose, and recover from errors',
 'Help and documentation',
 'Heuristic 1',
 'Heuristic 2',
 'Heuristic 3',
 'Heuristic 4',
 'Heuristic 5',
 'Heuristic 6',
 'Heuristic 7',
 'Heuristic 8',
 'Heuristic 9',
 'Heuristic 10',
 '1. Visibility of system status',
 '2. Match between system and the real world',
 '3. User control and freedom',
 '4. Consistency and standards',
 '5. Error prevention',
 '6. Recognition rather than recall',
 '7. Flexibility and efficiency of use',
 '8. Aesthetic and minimalist design',
 '9. Help users recognize, diagnose, and recover from errors',
 '10. Help and documentation',
 'Visibility of System Status',
 'Match Between System and the Real World',
 'User Control and Freedom',

In [133]:
catching[
    "issue_found"
].value_counts(
    dropna=False
)

issue_found
True     720
False    175
Name: count, dtype: int64

In [134]:
catching[
    "severity"
].value_counts(
    dropna=False
)

severity
2    306
1    251
0    174
3    155
4      9
Name: count, dtype: int64

In [135]:
evals_per_site = (
    catching[
        [
            "site",
            "evaluation"
        ]
    ]
    .drop_duplicates()
    .groupby("site")
    .size()
)

evals_per_site.value_counts()

3    30
Name: count, dtype: int64

In [136]:
VT_DIR = (
    PUBLIC_RAW
    / "03_29042675"
)

vt_metadata = pd.read_csv(
    VT_DIR
    / "Prototype_Metadata.csv"
)

vt_irr = pd.read_csv(
    VT_DIR
    / "IRR_Results.csv"
)

print(
    vt_metadata.shape,
    vt_irr.shape
)

display(
    vt_metadata
)

(20, 4) (104, 36)


,Prototype,Prototype Type,Domain,Scource URL
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0


In [137]:
display(
    vt_irr[
        [
            "Name",
            "Prototype Link",
            "H1 Compare",
            "H4 Compare",
            "H5 Compare",
            "H6 Compare"
        ]
    ].head(70)
)

,Name,Prototype Link,H1 Compare,H4 Compare,H5 Compare,H6 Compare
0,Food Order,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App-(Community)?node-id=0-1&p=f&t=Dx8Q70dOPPnSAAPO-0,1.0,1.0,1.0,1.0
1,Gaira (Community app),https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0,1.0,1.0,0.0,0.0
2,E-Tutor,https://www.figma.com/community/file/1271304360859402063,1.0,0.0,1.0,0.0
3,Tracking App,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,1.0,0.0,1.0,1.0
4,Travel,https://www.figma.com/community/file/1379689864793179441,1.0,1.0,0.0,1.0
5,Running,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,1.0,1.0,1.0,0.0
6,Bakery,https://www.figma.com/community/file/988854619809579450,1.0,0.0,1.0,1.0
7,Hospital 1,https://www.figma.com/community/file/1250314760992299521,1.0,1.0,1.0,1.0
8,Hospital 2,https://www.figma.com/community/file/12725823068122925,0.0,1.0,0.0,0.0
9,Housing,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,1.0,1.0,1.0,1.0


In [138]:
vt_metadata[
    "Domain"
].value_counts(
    dropna=False
)

Domain
Healthcare              5
Community               2
Education               2
Mobile app              2
Food Delivery           2
Delivery                1
Travel                  1
Food Delivey            1
Real Estate             1
Online Shoping          1
Blog                    1
Education (Quiz app)    1
Name: count, dtype: int64

In [139]:
vt_metadata[
    vt_metadata[
        "Domain"
    ]
    .astype(str)
    .str.contains(
        "educ",
        case=False,
        na=False
    )
]

,Prototype,Prototype Type,Domain,Scource URL
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063
17,True Art,Low Fidelity,Education,https://www.behance.net/gallery/165695079/True-Art-Education-App-UIUX-Case-Study/modules/934460805
18,Quizy,Low Fidelity,Education (Quiz app),https://www.behance.net/gallery/114288043/Quizzy-Mobile-Quiz-App/modules/652681113


In [140]:
ZENODO_DIR = (
    PUBLIC_RAW
    / "02_zenodo_21891322"
)

zenodo_nb_path = (
    ZENODO_DIR
    / "Heuristic evaluation (1).ipynb"
)

with open(
    zenodo_nb_path,
    "r",
    encoding="utf-8"
) as f:

    zenodo_nb = json.load(f)

print(
    "Celdas:",
    len(
        zenodo_nb["cells"]
    )
)

Celdas: 24


In [141]:
terms = [
    "H1",
    "H4",
    "H5",
    "H6",
    "heuristic",
    "expert",
    "ground",
    "violation",
    "severity"
]

matches = []

for i, cell in enumerate(
    zenodo_nb["cells"]
):

    text = "".join(
        cell.get(
            "source",
            []
        )
    )

    text_lower = text.lower()

    found = [
        term
        for term in terms
        if term.lower()
        in text_lower
    ]

    if found:

        matches.append({
            "cell": i,
            "terms": found,
            "text": text[:1000]
        })

pd.DataFrame(
    matches
)

,cell,terms,text
0,9,"[H1, H4, H5, H6, heuristic]","HEURISTICAS = [\n (""H1"", ""Visibilidade do status do sistema"",\n ""O design deve sempre manter os usuários informados sobre o que está acontecendo, por meio de feedback apropriado em tempo r..."
1,11,[heuristic],"def construir_prompt_zero_shot(dom_content=None):\n """"""\n Zero-shot: apenas a tarefa, as heurísticas e o conteúdo da interface.\n Sem exemplos e sem instruções de raciocínio intermediário..."
2,13,"[H1, heuristic]","EXEMPLO_FEW_SHOT = '''## Exemplo de avaliação heurística:\n\nInterface analisada: página de listagem de produtos em e-commerce.\n\nSaída esperada:\n[\n {\n ""heuristica_violada"": ""H1 — Visibili..."
3,15,[heuristic],"def construir_prompt_chain_of_thought(dom_content=None):\n """"""\n Chain-of-thought: instrui o modelo a raciocinar em etapas antes de produzir o output.\n Etapas: (1) identificar elementos,..."
4,23,[heuristic],"def exibir_resultados(todos_resultados):\n for nome_ui, estrategias in todos_resultados.items():\n print(f""\n{'#'*60}"")\n print(f"" 🖼️ UI: {nome_ui}"")\n print(f""{'#'*60}"")..."


In [142]:
P3M_DIR = (
    PUBLIC_RAW
    / "06_p3m"
)

p3m_all = pd.read_excel(
    P3M_DIR
    / "1. GPT-4 - All Possible Defects per Screenshot.xlsx"
)

p3m_consensus = pd.read_excel(
    P3M_DIR
    / "2. GPT-4 - Defects Consolidation.xlsx"
)

print(
    "All:",
    p3m_all.shape
)

print(
    "Consensus:",
    p3m_consensus.shape
)

print(
    p3m_all.columns.tolist()
)

print(
    p3m_consensus.columns.tolist()
)

All: (111, 7)
Consensus: (114, 9)
['Screenshot', 'Internal ID', 'Usability Problem', 'Violated Usability Heuristics', 'Severity', 'Explanation', 'Interface Reference']
['RED', 'FALSE POSITIVES', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']


In [143]:
display(
    p3m_all.head(30)
)

display(
    p3m_consensus.head(30)
)

,Screenshot,Internal ID,Usability Problem,Violated Usability Heuristics,Severity,Explanation,Interface Reference
0,1.0,D1.1,Inconsistency in the use of icons and text,Consistency and standards,2,"The interface uses icons and text in the side panel, where some commands are only icons while others have text, which may confuse users about the meaning of each button.",Sidebar tools
1,NaN,D1.2,Lack of feedback in interactive elements,Visibility of system status,3,"Interactive elements such as commodity category buttons do not provide visual feedback when clicked, making it difficult for users to know whether their action was acknowledged.","Buttons 'Metallic Minerals', 'Non-Metallic Minerals', etc."
2,NaN,D1.3,Information overload in the side panel,Aesthetic and minimalist design,2,"The side panel contains too many options that can overwhelm the user with unnecessary information, especially if some functions are not used frequently.",Side panel
3,NaN,D1.4,Lack of clarity in the function of zoom icons,Match between system and the real world,3,"The zoom icons (“+” and “-”) are standard, but lack information about their additional functionality, such as the current zoom level.",Zoom buttons in the upper right corner
4,NaN,D1.5,Uninformative page title 'Home',Recognition rather than recall,2,"The “Home” title does not adequately describe the context of the page, which may confuse users about its purpose when returning to it.",Upper section of the page
5,NaN,D1.6,Lack of clear indication of active selection,User control and freedom,3,"There is no clear indication of which options are actively selected or filtered, which can lead to errors when navigating the map.",Bottom navigation and filter bar
6,NaN,D1.7,Problem in accessing language settings intuitively,Match between system and the real world,2,"The “Portuguese” language option is located at the top with no clear indication that it is an interactive control, which can make navigation difficult for users who want to change the language.","Top of the page, next to the Portuguese language"
7,NaN,D1.8,Contact links are not prominent,Help and documentation,2,"The contact option is not prominent, making it difficult for users to find assistance when needed.",Contact and Information option at the top of the page
8,2.0,D2.1,Difficulty identifying icons in the side menu,Match between system and the real world,3,"The icons in the side menu may not be intuitive for all users, requiring more time to identify each function.",Side menu
9,NaN,D2.2,Lack of feedback or confirmation when selecting a commodity,"Visibility of system status, Error prevention",3,"The interface does not provide clear visual feedback when a commodity is selected, which can lead to uncertainty about the action performed.",Main Commodity Classes section


,RED,FALSE POSITIVES,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,BLUE,DEFECTS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Classification,Internal ID,Description,Explication,Interface reference,Is it a defect?,Reason,Verificação (LZ),NaN
3,Main,D1.1,Inconsistency in the use of icons and text,"A interface utiliza ícones e textos na barra lateral, onde alguns comandos estão apenas como ícones enquanto outros têm texto, o que pode confundir o usuário sobre o significado de cada botão.",Sidebar tools,No,"Looking at Screenshot 1, it looks like all the icons have text in the sidebar.",OK,NaN
4,Duplicate,D3.5,Inconsistency of icons and functions,Os ícones do painel lateral não possuem rótulos e podem não ter um significado claro ou consistente para todos os usuários.,Left side panel,No,NaN,OK,NaN
5,Duplicate,D4.2,Sidebar not very prominent,"O menu lateral tem ícones sem rótulos claros, o que pode causar confusão sobre sua função.",Left side menu.,No,NaN,OK,NaN
6,Duplicate,D7.2,Inconsistency of icons,"Os ícones na barra lateral não têm rótulos explicativos ou não são autoexplicativos, podendo confundir usuários que não reconheçam facilmente suas funções.",Left side bar,No,NaN,OK,NaN
7,Duplicate,D9.4,Consistency and standards,"The icons in the menu do not have visible descriptions, which can be confusing for users who do not understand what each icon means.",Left menu.,No,NaN,OK,NaN
8,Duplicate,D10.3,Consistency of menu icons,"The sidebar icons may not be intuitive for all users due to the lack of labels in some cases, which can cause ambiguity.",Left side menu,No,NaN,OK,NaN
9,Duplicate,D14.5,Sidebar with icons,The icons in the sidebar could have visible standard labels to make it easier for new users to understand.,Left side bar,No,NaN,OK,NaN


In [144]:
PHR_DIR = (
    PUBLIC_RAW
    / "07_phr"
)

with open(
    PHR_DIR / "README.txt",
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    phr_readme = f.read()

print(
    phr_readme
)

Overview

This dataset contains data from usability evaluations performed by 4 experts and energy consumption measurements obtained on a host machine while performing representative tasks on 4 PHR web portals. The dataset consists of 10 variables: 8 usability variables and 2 energy consumption variables. The energy consumption was obtained from each PHR and the usability results from each expert.
 
The data collection methodology was based on the completion of 20 tasks. The description of the tasks can be found in the paper. The experts rated a set of 8 heuristics on a Likert scale. The heuristics were based on those proposed by the author, Jakob Nielsen, and were specifically designed for PHRs. Each heuristic contained two questions as a checklist to guide the usability evaluation at the end of each task.
 
Energy consumption data was obtained using an energy meter connected to the power supply and monitor of a host machine under test. The device, known as Energy Efficient Tester (EET

In [145]:
for sheet in [
    "correlations",
    "t-student"
]:

    temp = pd.read_excel(
        PHR_DIR
        / "Results-PHRs-usab-sust.xlsx",
        sheet_name=sheet
    )

    print(
        "\nSHEET:",
        sheet
    )

    print(
        "Shape:",
        temp.shape
    )

    print(
        temp.columns.tolist()
    )

    display(
        temp.head(20)
    )


SHEET: correlations
Shape: (101, 51)
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VARIABLES IN THE STATISTICAL ANALISYS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,HEURISTICS (AUDITOR 1),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (AUDITOR 2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (AUDITOR 3),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (AUDITOR 4),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (MEAN VALUES),NaN,NaN,NaN,NaN,NaN,NaN,NaN,POWER DATA,NaN,NaN,NaN,HEALTHVET,NaN,MONITOR,PWSOURCE
2,NaN,NaN,NaN,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,MONITOR,PWSOURCE,NaN,H1,NaN,Correlation coefficient,0.12,0.3
3,NaN,HEALTHVET,T01,3,3,3,3,3,3,3,3,4,3,3,4,4,5,3,3,4,5,5,5,5,4,5,4,5,4,4,2,4,3,2,3,4,3.75,3.75,3.5,4,3.75,3.25,3.25,60.780581,185.302416,NaN,NaN,NaN,Sig. (bilateral),0.636,0.226
4,NaN,NaN,T02,3,3,3,3,3,3,3,3,4,3,3,4,4,5,3,3,4,5,5,4,5,4,4,4,4,5,4,3,4,5,4,4,3.75,4,3.75,3.5,4,4.25,3.5,3.5,62.046412,180.003197,NaN,NaN,NaN,N,18,18
5,NaN,NaN,T03,3,3,3,2,3,3,3,3,3,3,3,2,2,5,4,3,5,5,5,4,5,4,5,4,3,4,4,1,3,4,4,3,3.5,3.75,3.75,2.25,3.25,4,4,3.25,62.334192,180.528219,NaN,H2,NaN,Correlation coefficient,0.367,-0.193
6,NaN,NaN,T04,3,3,3,3,-,3,-,3,3,3,3,2,2,5,4,3,5,5,5,4,-,4,5,4,4,3,4,2,4,3,4,3,3.75,3.5,3.75,2.75,3,3.75,4.333333,3.25,61.024493,196.332188,NaN,NaN,NaN,Sig. (bilateral),0.134,0.443
7,NaN,NaN,T05,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,NaN,NaN,NaN,N,18,18
8,NaN,NaN,T06,3,3,3,3,3,3,3,3,4,3,3,3,3,5,3,3,5,5,5,5,3,4,4,4,4,3,3,2,4,4,4,4,4,3.5,3.5,3.25,3.25,4,3.5,3.5,59.527208,175.501902,NaN,H3,NaN,Correlation coefficient,0.159,-0.319
9,NaN,NaN,T07,3,3,3,3,4,4,4,4,4,3,3,3,3,5,3,4,3,5,4,4,5,4,5,4,3,3,5,3,3,3,4,3,3.25,3.5,3.75,3.25,3.75,4,4,3.75,58.083495,176.33451,NaN,NaN,NaN,Sig. (bilateral),0.529,0.198



SHEET: t-student
Shape: (104, 54)
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VARIABLES IN THE STATISTICAL ANALISYS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,HEURISTICS (ORDINAL VALUES),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (ORDINAL VALUES),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (ORDINAL VALUES),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (ORDINAL VALUES),NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEURISTICS (ORDINAL VALUES),NaN,NaN,NaN,NaN,NaN,NaN,NaN,POWER DATA,NaN,U: excellent usability / UN: regular usability,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,H01,H02,H03,H04,H05,H06,H07,H08,MONITOR,PWSOURCE,3.75,3.5,3.75,3.25,3.5,4,4,3.5,Thresholds
3,NaN,HEALTHVET,T01,3,3,3,3,3,3,3,3,4,3,3,4,4,5,3,3,4,5,5,5,5,4,5,4,5,4,4,2,4,3,2,3,4,3.75,3.75,3.5,4,3.75,3.25,3.25,60.780581,185.302416,U,U,U,U,U,NU,NU,NU,NaN
4,NaN,NaN,T02,3,3,3,3,3,3,3,3,4,3,3,4,4,5,3,3,4,5,5,4,5,4,4,4,4,5,4,3,4,5,4,4,3.75,4,3.75,3.5,4,4.25,3.5,3.5,62.046412,180.003197,U,U,U,U,U,U,NU,U,NaN
5,NaN,NaN,T03,3,3,3,2,3,3,3,3,3,3,3,2,2,5,4,3,5,5,5,4,5,4,5,4,3,4,4,1,3,4,4,3,3.5,3.75,3.75,2.25,3.25,4,4,3.25,62.334192,180.528219,NU,U,U,NU,NU,U,U,NU,NaN
6,NaN,NaN,T04,3,3,3,3,-,3,-,3,3,3,3,2,2,5,4,3,5,5,5,4,-,4,5,4,4,3,4,2,4,3,4,3,3.75,3.5,3.75,2.75,3,3.75,4.333333,3.25,61.024493,196.332188,U,U,U,NU,NU,NU,U,NU,NaN
7,NaN,NaN,T05,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,NaN
8,NaN,NaN,T06,3,3,3,3,3,3,3,3,4,3,3,3,3,5,3,3,5,5,5,5,3,4,4,4,4,3,3,2,4,4,4,4,4,3.5,3.5,3.25,3.25,4,3.5,3.5,59.527208,175.501902,U,U,NU,U,NU,U,NU,U,NaN
9,NaN,NaN,T07,3,3,3,3,4,4,4,4,4,3,3,3,3,5,3,4,3,5,4,4,5,4,5,4,3,3,5,3,3,3,4,3,3.25,3.5,3.75,3.25,3.75,4,4,3.75,58.083495,176.33451,NU,U,U,U,U,U,U,U,NaN


In [146]:
DASH_DIR = (
    PUBLIC_RAW
    / "08_dashboards"
)

dash_main = pd.read_csv(
    DASH_DIR
    / "dependent and independent variables.csv"
)

print(
    dash_main.shape
)

print(
    dash_main.columns.tolist()
)

display(
    dash_main.head(30)
)

(1008, 16)
['P', 'DB', 'Issue', 'Type', 'Q', 'Answer', 'Time', 'SGLT', 'mental', 'physical', 'temporal', 'performance', 'effort', 'frustration', 'avg_others', 'avg_all']


,P,DB,Issue,Type,Q,Answer,Time,SGLT,mental,physical,temporal,performance,effort,frustration,avg_others,avg_all
0,f118ebdd7e9350d704d9f91ee7de2702,3,Order&Grouping,Baseline,1,0,5.33,0,14.0,10.0,6,14,10,14.0,10.8,11.33
1,f118ebdd7e9350d704d9f91ee7de2702,2,InforOverload,Intervention,1,1,8.53,0,6.0,2.0,4,20,4,10.0,5.2,7.67
2,f118ebdd7e9350d704d9f91ee7de2702,1,InforOverload,Baseline,1,0,2.25,0,1.0,1.0,1,20,0,0.0,0.6,3.83
3,f118ebdd7e9350d704d9f91ee7de2702,3,Order&Grouping,Baseline,2,0,7.85,0,10.0,10.0,6,20,10,18.0,10.8,12.33
4,f118ebdd7e9350d704d9f91ee7de2702,7,DataUnderstanding,Baseline,2,1,8.82,0,4.0,4.0,2,8,5,14.0,5.8,6.17
5,f118ebdd7e9350d704d9f91ee7de2702,4,Order&Grouping,Intervention,2,1,8.67,0,6.0,10.0,2,20,4,3.0,5.0,7.50
6,f118ebdd7e9350d704d9f91ee7de2702,2,InforOverload,Intervention,2,1,5.80,0,16.0,13.0,6,20,6,10.0,10.2,11.83
7,f118ebdd7e9350d704d9f91ee7de2702,5,DataPresentation,Baseline,1,1,4.15,0,8.0,8.0,6,12,7,11.0,8.0,8.67
8,f118ebdd7e9350d704d9f91ee7de2702,6,DataPresentation,Intervention,1,1,2.10,0,2.0,2.0,2,20,2,0.0,1.6,4.67
9,f118ebdd7e9350d704d9f91ee7de2702,7,DataUnderstanding,Baseline,1,0,4.43,0,4.0,4.0,3,10,3,10.0,4.8,5.67


In [147]:
dash_zip = (
    DASH_DIR
    / "Interaction Data.zip"
)

with zipfile.ZipFile(
    dash_zip,
    "r"
) as z:

    real_jsons = [
        name
        for name in z.namelist()
        if (
            name.endswith(".json")
            and "__MACOSX" not in name
            and not Path(name).name.startswith("._")
        )
    ]

    print(
        "JSON reales:",
        len(real_jsons)
    )

    sample_name = real_jsons[0]

    raw_bytes = z.read(
        sample_name
    )

text = raw_bytes.decode(
    "utf-8-sig",
    errors="replace"
)

print(
    "Archivo:",
    sample_name
)

print(
    "Tamaño:",
    len(raw_bytes),
    "bytes"
)

print(
    "\nPrimeros 1000 caracteres:"
)

print(
    repr(text[:1000])
)

JSON reales: 50
Archivo: 1633946848251.json
Tamaño: 9991033 bytes

Primeros 1000 caracteres:
'{ "_id" : ObjectId("61640ce04622cf00272424fa"), "event" : "load", "screen" : { "sizeWidth" : 3440, "sizeHeight" : 1361, "resolutionWidth" : 3440, "resolutionHeight" : 1440, "htmlSizeWidth" : 3440, "htmlSizeHeight" : 1628, "usableSizeWidth" : 3440, "usableSizeHeight" : 1361 }, "client" : { "appName" : "Chrome", "version" : "93", "platform" : "MacIntel", "os" : "Mac/iOS" }, "episodeCount" : 1, "timestampms" : 1633946848252, "sessionstartms" : 1633946848251, "timezoneOffset" : -180, "sd" : "777", "sid" : null, "url" : "http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/", "urlFull" : "http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/" }\n{ "_id" : ObjectId("61640ce04622cf00272424fb"), "event" : "windowqueryfocus", "episodeCount" : 1, "timestampms" : 1633946848759, "sessionstartms" : 1633946848251, "timezoneOffset" : -180, "sd" : "777", "sid" : null, "ur

In [148]:
json_validation = []

with zipfile.ZipFile(
    dash_zip,
    "r"
) as z:

    for name in real_jsons:

        raw = z.read(name)

        text = raw.decode(
            "utf-8-sig",
            errors="replace"
        )

        try:

            data = json.loads(
                text
            )

            json_validation.append({
                "file": name,
                "valid_json": True,
                "error": None,
                "first_100_chars":
                    text[:100]
            })

        except json.JSONDecodeError as e:

            json_validation.append({
                "file": name,
                "valid_json": False,
                "error": str(e),
                "first_100_chars":
                    text[:100]
            })

dash_json_validation = (
    pd.DataFrame(
        json_validation
    )
)

print(
    dash_json_validation[
        "valid_json"
    ].value_counts(
        dropna=False
    )
)

display(
    dash_json_validation.head(20)
)

valid_json
False    50
Name: count, dtype: int64


,file,valid_json,error,first_100_chars
0,1633946848251.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""61640ce04622cf00272424fa""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 3440, """
1,1634049516613.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""61659ded4622cf0027251965""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1920, """
2,1635153366300.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""617675d74622cf002732cd81""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 2506, """
3,1634637167060.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""616e956f4622cf0027270bad""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1920, """
4,1633852258325.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""61629b624622cf002723f89a""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1440, """
5,1636971248412.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""619232f04622cf002739a70c""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1536, """
6,1635877381217.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""618182054622cf002735ebe2""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1536, """
7,1635529899496.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""617c34ac4622cf002734a55e""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1920, """
8,1634902121403.json,False,Expecting value: line 1 column 11 (char 10),"{ ""_id"" : ObjectId(""6172a06a4622cf00272fa972""), ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1280, """
9,1632586286250.json,False,Extra data: line 2 column 1 (char 675),"{ ""evID"" : ""614f4a2e4622cf00272198b7"", ""event"" : ""load"", ""screen"" : { ""sizeWidth"" : 1920, ""sizeHeigh"


In [149]:
RICO_DIR = (
    PUBLIC_RAW
    / "10_uicrit_rico"
)

uicrit = pd.read_csv(
    RICO_DIR
    / "uicrit_public.csv"
)

ui_details = pd.read_csv(
    RICO_DIR
    / "ui_details.csv"
)

app_details = pd.read_csv(
    RICO_DIR
    / "app_details.csv"
)

print(
    "UICrit:",
    uicrit.shape
)

print(
    "UI details:",
    ui_details.shape
)

print(
    "App details:",
    app_details.shape
)

display(
    uicrit.head()
)

UICrit: (2981, 9)
UI details: (66261, 4)
App details: (9384, 8)


,rico_id,task,aesthetics_rating,learnability,efficency,usability_rating,design_quality_rating,comments_source,comments
0,10089,Follow events to keep track on your favorite games/teams,6,3.0,3,6,6,"['human', 'human', 'human', 'human', 'both', 'both', 'both']","[""Comment 1\nThe expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the texts (Men's Tennis / Rome, Ga.) are too small a..."
1,10089,Explore the ACC Tournament schedules and match locations for the North Carolina team.,6,2.0,3,5,6,"['human', 'human', 'human', 'human', 'human', 'human', 'both']","['Comment 1\nThe expected standard is to make the most important information visually dominant.\nIn the current design, the Menu option is not visually prominent.\nTo fix this, we can enlarge the ..."
2,10089,Check the sport's schedule .,7,4.0,4,7,7,"['human', 'human', 'human', 'both']","['LLM Comment 1\nThe expected standard is for UI elements, such as the menu icon, to be fully visible and properly positioned within the screen boundaries.\n\nIn this current design, the menu icon..."
3,10112,"Click ""Register"" or ""Login"" to join Eskimi.",7,3.0,4,7,7,"['human', 'human', 'llm', 'llm', 'llm']","['Comment 1\nThe expected standard is that the design should be visually appealing and easy to use.\nIn the current design,the current design features a blurred background and The blurred backgrou..."
4,10112,Register or login to Eskimi,6,3.0,3,6,6,"['human', 'both']","['Comment 1\nThe expected tandard is to ensure sufficient contrast between buttons and their background for clear visibility and user interaction.\n\nIn the current design, the login button has lo..."


In [150]:
uicrit[
    [
        "rico_id",
        "aesthetics_rating",
        "learnability",
        "efficency",
        "usability_rating",
        "design_quality_rating"
    ]
].describe()

,rico_id,aesthetics_rating,learnability,efficency,usability_rating,design_quality_rating
count,2981.000000,2981.00000,2980.000000,2981.000000,2981.000000,2981.000000
mean,35870.717544,5.74740,3.009396,3.019457,5.815834,5.816169
std,20925.925638,1.08603,0.610894,0.651501,1.063723,1.019156
min,15.000000,1.00000,1.000000,1.000000,1.000000,1.000000
25%,17259.000000,5.00000,3.000000,3.000000,5.000000,5.000000
50%,36217.000000,6.00000,3.000000,3.000000,6.000000,6.000000
75%,53440.000000,6.00000,3.000000,3.000000,6.000000,6.000000
max,72194.000000,10.00000,5.000000,5.000000,10.000000,9.000000


In [151]:
uicrit_ids = set(
    pd.to_numeric(
        uicrit["rico_id"],
        errors="coerce"
    )
    .dropna()
    .astype(int)
)

rico_ids = set(
    pd.to_numeric(
        ui_details["UI Number"],
        errors="coerce"
    )
    .dropna()
    .astype(int)
)

print(
    "UICrit IDs:",
    len(uicrit_ids)
)

print(
    "UICrit encontrados en RICO:",
    len(
        uicrit_ids
        &
        rico_ids
    )
)

print(
    "Cobertura:",
    len(
        uicrit_ids
        &
        rico_ids
    )
    /
    len(uicrit_ids)
    * 100
)

UICrit IDs: 1000
UICrit encontrados en RICO: 1000
Cobertura: 100.0


In [152]:
candidate_audit[
    "unit_of_analysis"
] = None

candidate_audit[
    "has_native_X"
] = None

candidate_audit[
    "has_native_Y"
] = None

candidate_audit[
    "H1"
] = None

candidate_audit[
    "H4"
] = None

candidate_audit[
    "H5"
] = None

candidate_audit[
    "H6"
] = None

candidate_audit[
    "label_source"
] = None

candidate_audit[
    "label_quality"
] = None

candidate_audit[
    "join_key"
] = None

candidate_audit[
    "same_instance_XY"
] = None

candidate_audit[
    "decision_reason"
] = None

In [153]:
candidate_audit.to_csv(
    REPORTS_DIR
    / "candidate_dataset_audit.csv",
    index=False
)

In [154]:
import re

OBJECT_ID_RE = re.compile(
    r'ObjectId\("([^"]+)"\)'
)

ISO_DATE_RE = re.compile(
    r'ISODate\("([^"]+)"\)'
)

NUMBER_LONG_RE = re.compile(
    r'NumberLong\("?(-?\d+)"?\)'
)


def parse_mongo_export_line(line):

    line = line.strip()

    if not line:
        return None

    # MongoDB ObjectId -> string JSON
    line = OBJECT_ID_RE.sub(
        r'"\1"',
        line
    )

    # Por si aparecen fechas MongoDB
    line = ISO_DATE_RE.sub(
        r'"\1"',
        line
    )

    # Por si aparecen NumberLong
    line = NUMBER_LONG_RE.sub(
        r'\1',
        line
    )

    return json.loads(line)

In [155]:
dash_event_records = []
dash_parse_errors = []
dash_keys = set()

with zipfile.ZipFile(
    dash_zip,
    "r"
) as z:

    for name in real_jsons:

        with z.open(name) as f:

            for line_number, raw_line in enumerate(
                f,
                start=1
            ):

                text = raw_line.decode(
                    "utf-8-sig",
                    errors="replace"
                )

                try:

                    obj = parse_mongo_export_line(
                        text
                    )

                    if obj is None:
                        continue

                    dash_keys.update(
                        obj.keys()
                    )

                    dash_event_records.append({
                        "source_file": name,
                        "line_number": line_number,
                        "event": obj.get("event"),
                        "timestampms": obj.get("timestampms"),
                        "sessionstartms": obj.get("sessionstartms"),
                        "episodeCount": obj.get("episodeCount"),
                        "sd": obj.get("sd"),
                        "sid": obj.get("sid"),
                        "url": obj.get("url"),
                        "urlFull": obj.get("urlFull")
                    })

                except Exception as e:

                    dash_parse_errors.append({
                        "source_file": name,
                        "line_number": line_number,
                        "error": str(e),
                        "sample": text[:200]
                    })

In [156]:
dash_events = pd.DataFrame(
    dash_event_records
)

dash_errors = pd.DataFrame(
    dash_parse_errors
)

print(
    "Eventos parseados:",
    len(dash_events)
)

print(
    "Errores:",
    len(dash_errors)
)

print(
    "Archivos:",
    dash_events[
        "source_file"
    ].nunique()
)

print(
    "\nKeys top-level encontradas:"
)

print(
    sorted(dash_keys)
)

Eventos parseados: 1114761
Errores: 0
Archivos: 50

Keys top-level encontradas:
['_id', 'change', 'client', 'delta', 'episodeCount', 'evID', 'event', 'key', 'mobileTouch', 'mouse', 'node', 'screen', 'scroll', 'sd', 'select', 'sessionstartms', 'sid', 'timestampms', 'timezoneOffset', 'url', 'urlFull']


In [157]:
display(
    dash_events.head(20)
)

display(
    dash_events[
        [
            "source_file",
            "sessionstartms",
            "sd",
            "sid"
        ]
    ]
    .drop_duplicates()
    .head(50)
)

,source_file,line_number,event,timestampms,sessionstartms,episodeCount,sd,sid,url,urlFull
0,1633946848251.json,1,load,1633946848252,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
1,1633946848251.json,2,windowqueryfocus,1633946848759,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
2,1633946848251.json,3,mouseover,1633946853837,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
3,1633946848251.json,4,mouseout,1633946853874,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
4,1633946848251.json,5,mouseover,1633946853874,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
5,1633946848251.json,6,mouseout,1633946853890,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
6,1633946848251.json,7,mouseover,1633946853891,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
7,1633946848251.json,8,mouseout,1633946853907,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
8,1633946848251.json,9,mouseover,1633946853907,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/
9,1633946848251.json,10,mouseout,1633946853923,1633946848251,1,777,NaN,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/


,source_file,sessionstartms,sd,sid
0,1633946848251.json,1633946848251,777,NaN
15242,1634049516613.json,1634049516613,777,null
33770,1635153366300.json,1635153366300,777,null
65080,1634637167060.json,1634637167060,777,NaN
85211,1633852258325.json,1633852258325,777,null
96460,1636971248412.json,1636971248412,777,NaN
115425,1635877381217.json,1635877381217,777,NaN
145080,1635529899496.json,1635529899496,777,null
157051,1634902121403.json,1634902121403,777,NaN
179130,1632586286250.json,1632586286250,777,NaN


In [158]:
dash_events[
    "event"
].value_counts(
    dropna=False
).head(50)

event
mouseover           385267
mouseout            379054
keydown              75974
keyup                65983
keypress             55334
mouseup              36196
change               34512
mousedown            31156
scroll               30847
mousewheel           14390
select                1981
dblclick              1158
mobileTouchEnd         826
mobileTouchStart       763
windowqueryfocus       540
windowqueryblur        504
contextmenu            180
load                    50
copy                    24
cut                     20
paste                    2
Name: count, dtype: int64

In [159]:
def normalize_heuristic(value):

    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    # Eliminar prefijos como:
    # "1. Visibility..."
    text = re.sub(
        r"^\d+\.\s*",
        "",
        text
    )

    direct_map = {
        "visibility of system status": "H1",
        "consistency and standards": "H4",
        "error prevention": "H5",
        "recognition rather than recall": "H6"
    }

    if text in direct_map:
        return direct_map[text]

    # Variantes "Heuristic 1", etc.
    match = re.fullmatch(
        r"heuristic\s*(\d+)",
        text
    )

    if match:

        number = int(
            match.group(1)
        )

        if number in {
            1,
            4,
            5,
            6
        }:
            return f"H{number}"

    return None

In [160]:
catching[
    "heuristic_code"
] = catching[
    "heuristic"
].apply(
    normalize_heuristic
)

In [161]:
catching[
    "heuristic_code"
].value_counts(
    dropna=False
)

heuristic_code
NaN    537
H5      90
H6      90
H1      89
H4      89
Name: count, dtype: int64

In [162]:
catching_selected = catching[
    catching[
        "heuristic_code"
    ].isin(
        [
            "H1",
            "H4",
            "H5",
            "H6"
        ]
    )
].copy()

print(
    catching_selected.shape
)

pd.crosstab(
    catching_selected[
        "heuristic_code"
    ],
    catching_selected[
        "issue_found"
    ]
)

(358, 9)


issue_found,False,True
heuristic_code,,
H1,4,85
H4,19,70
H5,12,78
H6,35,55


In [163]:
catching_coverage = (
    catching_selected
    .groupby(
        [
            "site",
            "evaluation"
        ]
    )[
        "heuristic_code"
    ]
    .nunique()
)

catching_coverage.value_counts()

heuristic_code
4    88
3     2
Name: count, dtype: int64

In [164]:
catching_y = (
    catching_selected
    .groupby(
        [
            "site",
            "evaluation",
            "heuristic_code"
        ]
    )
    .agg(
        violation=(
            "issue_found",
            "max"
        ),
        max_severity=(
            "severity",
            "max"
        )
    )
    .reset_index()
)

display(
    catching_y.head(30)
)

,site,evaluation,heuristic_code,violation,max_severity
0,Business-Website,eval1,H1,True,2
1,Business-Website,eval1,H4,True,1
2,Business-Website,eval1,H5,True,2
3,Business-Website,eval1,H6,True,1
4,Business-Website,eval2,H1,True,3
5,Business-Website,eval2,H4,True,1
6,Business-Website,eval2,H5,True,2
7,Business-Website,eval2,H6,True,1
8,Business-Website,eval3,H1,True,3
9,Business-Website,eval3,H4,True,1
